# ArmorVault — Stage 2: MiniCPM-V only

Run this in a fresh Colab runtime after Stage 1 saved files to Google Drive. PaddleOCR is not installed or loaded here, leaving GPU memory for MiniCPM-V 4.5 AWQ.

In [ ]:
import subprocess
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
work = Path('/content/drive/MyDrive/armorvault-ocr-lab')
image_path = work / 'public_demo.png'
extraction_path = work / 'paddle-extraction.txt'
if not image_path.exists() or not extraction_path.exists():
    raise FileNotFoundError('Run Stage 1 first and confirm its final cell saved files to Drive.')
print('Found Stage 1 output.')

In [ ]:
%pip install -q 'transformers>=4.51,<5' 'accelerate>=1.4' 'autoawq>=0.2.9' pillow
print('MiniCPM dependencies installed.')

In [ ]:
import json, re, time, torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer
model_name = 'openbmb/MiniCPM-V-4_5-AWQ'
extraction = extraction_path.read_text(encoding='utf-8', errors='replace')[:60000]
started = time.perf_counter()
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, device_map='auto', low_cpu_mem_usage=True).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
init_seconds = time.perf_counter() - started
prompt = '''Use the document image and PaddleOCR-VL extraction below. Return JSON only with exactly these keys: documentType, documentNumber, holderName, issueDate, expiryDate, totalAmount, currency, mrzLines. Use null for uncertain scalar values and [] for absent MRZ lines. Dates must be YYYY-MM-DD. Never invent unreadable characters.\n\nPaddleOCR-VL extraction:\n''' + extraction
started = time.perf_counter()
answer = model.chat(msgs=[{'role': 'user', 'content': [Image.open(image_path).convert('RGB'), prompt]}], tokenizer=tokenizer, enable_thinking=False)
inference_seconds = time.perf_counter() - started
cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', answer.strip(), flags=re.I | re.S)
try:
    fields = json.loads(cleaned)
except json.JSONDecodeError:
    start, end = cleaned.find('{'), cleaned.rfind('}')
    if start < 0 or end <= start:
        raise RuntimeError('MiniCPM did not return a JSON object.')
    fields = json.loads(cleaned[start:end + 1])
result = {'model': model_name, 'initializationSeconds': round(init_seconds, 2), 'inferenceSeconds': round(inference_seconds, 2), 'fields': fields}
(work / 'stage2-result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(result, ensure_ascii=False, indent=2))

The final JSON is saved at `Google Drive/armorvault-ocr-lab/stage2-result.json`.